# Coach House user audience validation

Reproducible checks for the sanitized August 3, 2026 aggregate snapshot. No personal identifiers are loaded.

## Visualization plan

The partner report uses a stacked monthly signup bar chart to answer when the external account base arrived and how much was regular versus tester traffic. A table compares Build, Find, and other selected focus paths across registration, sign-in, activity, organization ownership, and payment.

In [ ]:
import json
from pathlib import Path

snapshot_path = Path('analysis-snapshot.json')
data = json.loads(snapshot_path.read_text())
summary = data['account_summary']
summary

In [ ]:
external = summary['external_nonsynthetic_accounts']
assert summary['total_auth_records'] - summary['synthetic_demo_accounts'] - summary['internal_staff_accounts'] == external
assert summary['regular_accounts'] + summary['tester_accounts'] == external
assert summary['ever_signed_in'] + summary['never_signed_in'] == external
assert sum(row['accounts'] for row in data['signup_months']) == external
assert sum(row['accounts'] for row in data['selected_focus_segments']) == external
assert sum(row['accounts'] for row in data['email_mix']) == external
assert sum(row['organizations'] for row in data['organization_formation_stages']) == summary['organization_owners']
print('All cohort reconciliation checks passed.')

In [ ]:
headline = {
    'ever_signed_in_share': summary['ever_signed_in'] / external,
    'active_30d_share': summary['tracked_active_30d'] / external,
    'current_paying_regular_share': data['payment_summary']['current_paying_regular_users'] / external,
    'may_signup_share': data['may_2026_cohort']['accounts'] / external,
    'organization_owner_share': summary['organization_owners'] / external,
}
{key: f'{value:.1%}' for key, value in headline.items()}

## Interpretation guardrails

The 180 records are accounts, not deduplicated humans or active customers. Signup source is unknown because historical first-touch attribution was not stored. Vercel traffic is anonymous and cannot be joined to Supabase accounts. Stripe is the authority for the current payer count.